# Static Feature Extraction

Extract track-level spectral features (MFCC, chroma, spectral descriptors, ZCR, RMS, tempo) with mean/std/min/max/median aggregations.

**Input:** `data/processed/static_labels.parquet` (from notebook 01)  
**Output:** `data/features/static_features.parquet`

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.features.spectral_features import (
    extract_static_features_dataset,
    extract_track_spectral_features,
    save_static_features,
)

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

labels_path = resolve_path(root, configs["paths"]["processed"]["static_labels"])
labels_df = pd.read_parquet(labels_path)
print(f"Tracks with labels: {len(labels_df)}")
labels_df.head()

Tracks with labels: 1802


,song_id,valence,arousal,audio_path,has_static_annotation,has_dynamic_annotation,has_audio,valence_threshold,arousal_threshold,emotion_quadrant,emotion_class
0,2,3.1,3.0,C:\Users\athen\Desktop\music-emotion-recogniti...,True,True,True,4.9,4.9,Q4,low_valence_low_arousal
1,3,3.5,3.3,C:\Users\athen\Desktop\music-emotion-recogniti...,True,True,True,4.9,4.9,Q4,low_valence_low_arousal
2,4,5.7,5.5,C:\Users\athen\Desktop\music-emotion-recogniti...,True,True,True,4.9,4.9,Q1,high_valence_high_arousal
3,5,4.4,5.3,C:\Users\athen\Desktop\music-emotion-recogniti...,True,True,True,4.9,4.9,Q3,low_valence_high_arousal
4,7,5.8,6.4,C:\Users\athen\Desktop\music-emotion-recogniti...,True,True,True,4.9,4.9,Q1,high_valence_high_arousal


## Extract features for all tracks

In [2]:
features_df = extract_static_features_dataset(labels_df, configs)
print(f"Extracted features for {len(features_df)} tracks")
print(f"Number of feature columns: {len(features_df.columns) - 1}")

features_df.head()

Extracting static features:   0%|          | 0/1802 [00:00<?, ?it/s]

Extracted features for 1802 tracks
Number of feature columns: 186


,song_id,chroma_10_max,chroma_10_mean,chroma_10_median,chroma_10_min,chroma_10_std,chroma_11_max,chroma_11_mean,chroma_11_median,chroma_11_min,...,spectral_rolloff_1_mean,spectral_rolloff_1_median,spectral_rolloff_1_min,spectral_rolloff_1_std,tempo_bpm,zcr_1_max,zcr_1_mean,zcr_1_median,zcr_1_min,zcr_1_std
0,2,1.0,0.662667,0.790196,0.012360,0.352839,1.0,0.469289,0.450577,0.012454,...,3797.049219,3639.111328,1701.123047,1149.941753,143.554688,0.191406,0.068091,0.064941,0.013184,0.026750
1,3,1.0,0.435158,0.302385,0.025371,0.324933,1.0,0.392531,0.337649,0.038061,...,2072.692896,1582.690430,204.565430,1372.498961,95.703125,0.140137,0.024278,0.020508,0.005859,0.016115
2,4,1.0,0.755609,1.000000,0.011561,0.335549,1.0,0.431298,0.438582,0.015431,...,3365.961537,2917.749023,785.961914,1269.282454,172.265625,0.294922,0.067320,0.060059,0.013672,0.033544
3,5,1.0,0.129347,0.059378,0.000698,0.175071,1.0,0.298784,0.240181,0.000841,...,3276.060413,3273.046875,559.863281,1389.726767,99.384014,0.375977,0.064114,0.059570,0.016602,0.035028
4,7,1.0,0.271460,0.173977,0.005444,0.286505,1.0,0.334192,0.183820,0.003469,...,3022.181176,3003.881836,204.565430,1680.652704,117.453835,0.140137,0.022924,0.018066,0.007324,0.015993


## Save features

In [3]:
saved_paths = save_static_features(features_df, configs)
saved_paths

{'parquet': WindowsPath('C:/Users/athen/Desktop/music-emotion-recognition/data/features/static_features.parquet'),
 'csv': WindowsPath('C:/Users/athen/Desktop/music-emotion-recognition/data/features/static_features.csv')}

## Optional: inspect a single track

In [4]:
sample = labels_df.iloc[0]
sample_features = extract_track_spectral_features(sample["audio_path"], configs)
pd.Series(sample_features).head(10)

mfcc_1_mean     -138.777924
mfcc_1_std        48.472599
mfcc_1_min      -563.732178
mfcc_1_max       -20.263084
mfcc_1_median   -129.765137
mfcc_2_mean      115.720337
mfcc_2_std        24.211376
mfcc_2_min         0.000000
mfcc_2_max       170.752426
mfcc_2_median    118.306320
dtype: float64